In [ ]:
"""
LLPS Peptide Classification with Lipidation-Aware Fingerprints
Enhanced with: Multi-Model Comparison, ROC Curves, Confusion Matrices
Complete workflow: Training → Multi-Model Comparison → Visualizations
"""

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    # File paths
    #Replace with actual file path location
    'output_dir':  "data/output",
    'training_file':  "data/input.xlsx",
    
    # Training parameters
    'random_state': 42,
    'test_size': 0.2,
    'n_splits': 5,
    'n_estimators': 500,
    'num_leaves': 31,
    'learning_rate': 0.05,
    
    # Lipidation filter
    'lipidation_filter': 'all',
    
    # Colors
    'colors': {'soluble': '#2ecc71', 'llps': '#f39c12', 'aggregate': '#e74c3c'},
    
    # Category mapping
    'aggregate_categories': ['aggregate', 'LLPS'],
    'soluble_categories': ['soluble'],
}

# ============================================================================
# IMPORTS
# ============================================================================

import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, List, Dict
import pickle

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, matthews_corrcoef, accuracy_score,
    roc_curve, auc, confusion_matrix, precision_recall_curve,
    precision_score, recall_score, balanced_accuracy_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from skfp.preprocessing import MolFromAminoseqTransformer
from skfp.fingerprints import RDKitFingerprint
from lightgbm import LGBMClassifier

try:
    import xgboost as xgb
    HAS_XGB = True
except:
    HAS_XGB = False
    print("[WARNING] XGBoost not installed")

from rdkit import Chem

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ============================================================================
# PART 1: ROC CURVE VISUALIZATION
# ============================================================================

class ROCVisualization:
    """Visualize ROC curves for multi-class classification."""
    
    def __init__(self, colors=None):
        self.colors = colors or sns.color_palette("husl", 10)
    
    def plot_macro_roc_curve(self, y_true, y_pred_proba, class_names, 
                            title, output_path, dpi=300):
        """Plot macro-averaged ROC curve for multi-class classification."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fpr_list = []
        tpr_list = []
        auc_per_class = {}
        
        for i in range(n_classes):
            if y_true_bin[:, i].sum() == 0:
                print(f"    ⚠️  No positive samples for class {class_names[i]}")
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            auc_per_class[class_names[i]] = roc_auc
            fpr_list.append(fpr)
            tpr_list.append(tpr)
        
        mean_fpr = np.linspace(0, 1, 100)
        tpr_interp_list = []
        
        for fpr, tpr in zip(fpr_list, tpr_list):
            tpr_interp = np.interp(mean_fpr, fpr, tpr)
            tpr_interp_list.append(tpr_interp)
        
        mean_tpr = np.mean(tpr_interp_list, axis=0)
        mean_tpr[0], mean_tpr[-1] = 0, 1
        macro_auc = auc(mean_fpr, mean_tpr)
        
        fig, ax = plt.subplots(figsize=(10, 8), dpi=dpi)
        
        ax.plot(mean_fpr, mean_tpr, 
                label=f'Macro-avg ROC (AUC = {macro_auc:.3f})', 
                lw=3.5, color='darkblue')
        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
        ax.set_title(title, fontsize=15, fontweight='bold')
        ax.legend(loc='lower right', fontsize=12)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        roc_csv_filename = output_path.stem.replace('roc_', 'roc_data_') + '.csv'
        roc_csv_path = output_path.parent / roc_csv_filename
        roc_df = pd.DataFrame({
            'FPR': mean_fpr,
            'TPR': mean_tpr,
            'Macro_AUC': macro_auc
        })
        roc_df.to_csv(roc_csv_path, index=False)
        
        print(f"    ✓ Macro ROC: {output_path.name}")
        print(f"    ✓ ROC data: {roc_csv_path.name}")
        
        return mean_fpr, mean_tpr, macro_auc, auc_per_class
    
    def plot_per_class_roc_curves(self, y_true, y_pred_proba, class_names,
                                 title, output_path, dpi=300):
        """Plot per-class (One-vs-Rest) ROC curves."""
        
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
        
        fig, axes = plt.subplots(1, n_classes, figsize=(6*n_classes, 6), dpi=dpi)
        if n_classes == 1:
            axes = [axes]
        
        colors_palette = sns.color_palette("husl", n_classes)
        
        for i, class_name in enumerate(class_names):
            ax = axes[i]
            
            if y_true_bin[:, i].sum() == 0:
                ax.text(0.5, 0.5, f'No positive samples\nfor {class_name}',
                       ha='center', va='center', fontsize=12, transform=ax.transAxes)
                continue
            
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            
            ax.plot(fpr, tpr, color=colors_palette[i], linewidth=3,
                   label=f'{class_name}\n(AUC = {roc_auc:.3f})')
            ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
            
            ax.set_xlim([0.0, 1.0])
            ax.set_ylim([0.0, 1.05])
            ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
            ax.set_title(f'ROC: {class_name} (One-vs-Rest)', fontsize=12, fontweight='bold')
            ax.legend(loc='lower right', fontsize=11)
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ Per-class ROC: {output_path.name}")


# ============================================================================
# PART 2: LIPIDATION-AWARE FINGERPRINTER
# ============================================================================

class LipidationAwareFingerprinter:
    """Compute fingerprints with lipidation as chemical modification."""
    
    LIPIDATION_SMILES = {
        'octanoyl': 'CCCCCCC',
        'hexanoyl': 'CCCCC',
    }
    
    def __init__(self):
        self.mol_transformer = MolFromAminoseqTransformer(n_jobs=-1)
        self.fp_transformer = RDKitFingerprint(n_jobs=-1)

    def preprocess_for_lipidation(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize lipidation names and apply filter."""
        df_clean = df.copy()
        
        if 'lipidation' in df_clean.columns:
            df_clean['lipidation'] = df_clean['lipidation'].fillna('none')
            df_clean['lipidation'] = df_clean['lipidation'].str.lower().str.strip()
        else:
            df_clean['lipidation'] = 'none'
        
        print(f"  Original: {len(df_clean)} samples")
        print(f"  Lipidation distribution (before filter):")
        for lip_type, count in df_clean['lipidation'].value_counts().items():
            print(f"    {lip_type}: {count}")
        
        lipidation_filter = CONFIG['lipidation_filter'].lower()
        
        if lipidation_filter == 'lipidated_only':
            print(f"\n  ⚠️  FILTER: Lipidated only (excluding 'none')")
            df_clean = df_clean[df_clean['lipidation'] != 'none'].copy()
        
        elif lipidation_filter == 'non_lipidated_only':
            print(f"\n  ⚠️  FILTER: Non-lipidated only (keeping 'none' only)")
            df_clean = df_clean[df_clean['lipidation'] == 'none'].copy()
        
        elif lipidation_filter == 'all':
            print(f"\n  ✓ FILTER: All peptides (lipidated + non-lipidated)")
        
        valid_lipidations = ['none', 'octanoyl', 'hexanoyl']
        valid_mask = df_clean['lipidation'].isin(valid_lipidations)
        if (~valid_mask).sum() > 0:
            print(f"  Excluding {(~valid_mask).sum()} unknown lipidation types")
            df_clean = df_clean[valid_mask].copy()
        
        print(f"  Final after filter: {len(df_clean)} samples")
        print(f"  Final distribution:")
        for lip_type, count in df_clean['lipidation'].value_counts().items():
            print(f"    {lip_type}: {count}")
        
        return df_clean

    def create_lipidated_smiles(self, peptide_smiles: str, lipidation: str) -> str:
        """Add lipidation as amide bond to N-terminus amino acid."""
        lipidation = lipidation.lower().strip()
        
        if lipidation == 'none' or lipidation not in self.LIPIDATION_SMILES:
            return peptide_smiles
        
        lipid_chain = self.LIPIDATION_SMILES[lipidation]
        pattern = '[C@@H](N)'
        replacement = f'[C@@H](NC(=O){lipid_chain})'
        
        lipidated_smiles = peptide_smiles.replace(pattern, replacement)
        return lipidated_smiles

    def compute_fingerprints_with_lipidation(self, sequences: np.ndarray, 
                                            lipidations: np.ndarray) -> np.ndarray:
        """Compute fingerprints with lipidation as chemical modification."""
        print("  Converting sequences...")
        mols = self.mol_transformer.transform(sequences)
        mols = np.array(mols)
        
        print("  Adding lipidation modifications...")
        modified_mols = []
        for i, (mol, lipidation) in enumerate(zip(mols, lipidations)):
            if mol is None:
                modified_mols.append(None)
                continue
            try:
                seq_smiles = Chem.MolToSmiles(mol)
                lipidation_lower = lipidation.lower().strip() if isinstance(lipidation, str) else 'none'
                lipidated_smiles = self.create_lipidated_smiles(seq_smiles, lipidation_lower)
                
                lipidated_mol = Chem.MolFromSmiles(lipidated_smiles)
                modified_mols.append(lipidated_mol if lipidated_mol else mol)
            except Exception as e:
                modified_mols.append(mol)
        
        print("  Computing RDKit fingerprints...")
        X = self.fp_transformer.transform(np.array(modified_mols))
        print(f"  Fingerprint shape: {X.shape}")
        
        return X


# ============================================================================
# PART 3: MULTI-MODEL TRAINING ENGINE
# ============================================================================

class MultiModelTrainer:
    """Train and compare multiple models."""
    
    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.models_dict = {}
        self.metrics_list = []
        self.best_model_name = None
        self.best_model = None
        self.X_test = None
        self.y_test = None
        self.y_test_pred_dict = {}
        self.y_test_proba_dict = {}

    def train_models(self, X_train: np.ndarray, X_test: np.ndarray,
                    y_train: np.ndarray, y_test: np.ndarray, 
                    class_mapping: Dict) -> pd.DataFrame:
        """Train multiple models and compare."""
        
        print("\n" + "="*80)
        print("TRAINING MULTIPLE MODELS")
        print("="*80)
        
        self.X_test = X_test
        self.y_test = y_test
        
        # Logistic Regression
        print("\n• Logistic Regression...")
        try:
            clf_lr = LogisticRegression(max_iter=5000, random_state=self.random_state,
                                       class_weight='balanced', n_jobs=-1)
            clf_lr.fit(X_train, y_train)
            self.models_dict['LogisticRegression'] = clf_lr
            
            y_pred_lr = clf_lr.predict(X_test)
            y_proba_lr = clf_lr.predict_proba(X_test)
            self.y_test_pred_dict['LogisticRegression'] = y_pred_lr
            self.y_test_proba_dict['LogisticRegression'] = y_proba_lr
            
            metrics_lr = self._compute_metrics('LogisticRegression', y_pred_lr, y_proba_lr, y_test)
            self.metrics_list.append(metrics_lr)
            print(f"  Accuracy: {metrics_lr['Accuracy']:.4f}, AUROC: {metrics_lr['AUROC']:.4f}")
        except Exception as e:
            print(f"  Failed: {e}")
        
        # Random Forest
        print("\n• Random Forest...")
        try:
            clf_rf = RandomForestClassifier(n_estimators=100, random_state=self.random_state,
                                           class_weight='balanced', n_jobs=-1, max_depth=15)
            clf_rf.fit(X_train, y_train)
            self.models_dict['RandomForest'] = clf_rf
            
            y_pred_rf = clf_rf.predict(X_test)
            y_proba_rf = clf_rf.predict_proba(X_test)
            self.y_test_pred_dict['RandomForest'] = y_pred_rf
            self.y_test_proba_dict['RandomForest'] = y_proba_rf
            
            metrics_rf = self._compute_metrics('RandomForest', y_pred_rf, y_proba_rf, y_test)
            self.metrics_list.append(metrics_rf)
            print(f"  Accuracy: {metrics_rf['Accuracy']:.4f}, AUROC: {metrics_rf['AUROC']:.4f}")
        except Exception as e:
            print(f"  Failed: {e}")
        
        # Gradient Boosting
        print("\n• Gradient Boosting...")
        try:
            clf_gb = GradientBoostingClassifier(n_estimators=100, random_state=self.random_state,
                                              max_depth=5, learning_rate=0.1)
            clf_gb.fit(X_train, y_train)
            self.models_dict['GradientBoosting'] = clf_gb
            
            y_pred_gb = clf_gb.predict(X_test)
            y_proba_gb = clf_gb.predict_proba(X_test)
            self.y_test_pred_dict['GradientBoosting'] = y_pred_gb
            self.y_test_proba_dict['GradientBoosting'] = y_proba_gb
            
            metrics_gb = self._compute_metrics('GradientBoosting', y_pred_gb, y_proba_gb, y_test)
            self.metrics_list.append(metrics_gb)
            print(f"  Accuracy: {metrics_gb['Accuracy']:.4f}, AUROC: {metrics_gb['AUROC']:.4f}")
        except Exception as e:
            print(f"  Failed: {e}")
        
        # Support Vector Machine
        print("\n• Support Vector Machine...")
        try:
            clf_svm = SVC(kernel='rbf', C=1.0, probability=True,
                         class_weight='balanced', random_state=self.random_state)
            clf_svm.fit(X_train, y_train)
            self.models_dict['SVM'] = clf_svm
            
            y_pred_svm = clf_svm.predict(X_test)
            y_proba_svm = clf_svm.predict_proba(X_test)
            self.y_test_pred_dict['SVM'] = y_pred_svm
            self.y_test_proba_dict['SVM'] = y_proba_svm
            
            metrics_svm = self._compute_metrics('SVM', y_pred_svm, y_proba_svm, y_test)
            self.metrics_list.append(metrics_svm)
            print(f"  Accuracy: {metrics_svm['Accuracy']:.4f}, AUROC: {metrics_svm['AUROC']:.4f}")
        except Exception as e:
            print(f"  Failed: {e}")
        
        # LightGBM
        print("\n• LightGBM...")
        try:
            clf_lgb = LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05,
                                    class_weight='balanced', random_state=self.random_state,
                                    n_jobs=-1, verbose=-1, is_unbalance=True)
            clf_lgb.fit(X_train, y_train)
            self.models_dict['LightGBM'] = clf_lgb
            
            y_pred_lgb = clf_lgb.predict(X_test)
            y_proba_lgb = clf_lgb.predict_proba(X_test)
            self.y_test_pred_dict['LightGBM'] = y_pred_lgb
            self.y_test_proba_dict['LightGBM'] = y_proba_lgb
            
            metrics_lgb = self._compute_metrics('LightGBM', y_pred_lgb, y_proba_lgb, y_test)
            self.metrics_list.append(metrics_lgb)
            print(f"  Accuracy: {metrics_lgb['Accuracy']:.4f}, AUROC: {metrics_lgb['AUROC']:.4f}")
        except Exception as e:
            print(f"  Failed: {e}")
        
        # XGBoost
        if HAS_XGB:
            print("\n• XGBoost...")
            try:
                clf_xgb = xgb.XGBClassifier(n_estimators=100, random_state=self.random_state,
                                           scale_pos_weight=1, verbose=0, n_jobs=-1)
                clf_xgb.fit(X_train, y_train)
                self.models_dict['XGBoost'] = clf_xgb
                
                y_pred_xgb = clf_xgb.predict(X_test)
                y_proba_xgb = clf_xgb.predict_proba(X_test)
                self.y_test_pred_dict['XGBoost'] = y_pred_xgb
                self.y_test_proba_dict['XGBoost'] = y_proba_xgb
                
                metrics_xgb = self._compute_metrics('XGBoost', y_pred_xgb, y_proba_xgb, y_test)
                self.metrics_list.append(metrics_xgb)
                print(f"  Accuracy: {metrics_xgb['Accuracy']:.4f}, AUROC: {metrics_xgb['AUROC']:.4f}")
            except Exception as e:
                print(f"  Failed: {e}")
        
        metrics_df = pd.DataFrame(self.metrics_list).sort_values('Accuracy', ascending=False)
        
        # Select best model
        best_idx = metrics_df['Accuracy'].idxmax()
        self.best_model_name = metrics_df.iloc[best_idx]['Model']
        self.best_model = self.models_dict[self.best_model_name]
        
        print(f"\n🏆 BEST MODEL: {self.best_model_name}")
        print(f"   Accuracy: {metrics_df.iloc[best_idx]['Accuracy']:.4f}")
        
        return metrics_df
    
    def _compute_metrics(self, model_name: str, y_pred, y_proba, y_test) -> Dict:
        """Compute metrics for a model."""
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        mcc = matthews_corrcoef(y_test, y_pred)
        
        try:
            auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        except:
            auc_score = 0.0
        
        return {
            'Model': model_name,
            'Accuracy': acc,
            'Balanced_Accuracy': bal_acc,
            'Precision': prec,
            'Recall': rec,
            'F1_Score': f1,
            'MCC': mcc,
            'AUROC': auc_score,
        }


# ============================================================================
# PART 4: CLASSIFIER
# ============================================================================

class LLPSFingerprintClassifier:
    """Multiclass LLPS classifier with lipidation-aware fingerprints."""
    
    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.results = {}
        self.best_model = None
        self.class_mapping = {}
        self.lip_fingerprinter = LipidationAwareFingerprinter()
        self.roc_viz = ROCVisualization(CONFIG['colors'])

    def load_and_prepare_data(self, excel_path: str) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
        """Load and prepare data with lipidation preprocessing."""
        print("[1/4] Loading and preparing data...")
        df = pd.read_excel(excel_path)
        print(f"  Total rows: {len(df)}")
        
        df = self.lip_fingerprinter.preprocess_for_lipidation(df)
        
        df_clean = df[~df['category'].str.lower().str.contains('unknown', na=False)].copy()
        
        label_mapping = {}
        for cat in CONFIG['soluble_categories']:
            label_mapping[cat.lower()] = 0
        label_mapping['llps'] = 1
        for cat in CONFIG['aggregate_categories']:
            if cat.lower() != 'llps':
                label_mapping[cat.lower()] = 2
        
        df_clean['category_lower'] = df_clean['category'].str.lower()
        known_mask = df_clean['category_lower'].isin(label_mapping.keys())
        df_clean = df_clean[known_mask].copy()
        
        df_clean['label'] = df_clean['category_lower'].map(label_mapping)
        self.class_mapping = {v: k for k, v in label_mapping.items()}
        
        print(f"  Category distribution:")
        for label, name in self.class_mapping.items():
            count = (df_clean['label'] == label).sum()
            print(f"    {name.upper()}: {count}")
        
        sequences = df_clean['sequence'].values
        labels = df_clean['label'].values
        
        return sequences, labels, df_clean

    def compute_fingerprints_lipidation_aware(self, sequences: np.ndarray, 
                                             lipidations: np.ndarray) -> np.ndarray:
        """Compute lipidation-aware fingerprints."""
        print("\n[2/4] Computing lipidation-aware fingerprints...")
        X = self.lip_fingerprinter.compute_fingerprints_with_lipidation(sequences, lipidations)
        return X

    def train_with_holdout_multimodel(self, X: np.ndarray, y: np.ndarray, output_dir: Path) -> Dict:
        """Train with holdout test set and multi-model comparison."""
        print(f"\n[3/4] Training with holdout test set (80/20)...")
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=CONFIG['test_size'], stratify=y, random_state=CONFIG['random_state']
        )
        
        print(f"  Train: {len(X_train)}, Test: {len(X_test)}")
        
        # Train multiple models
        trainer = MultiModelTrainer(random_state=self.random_state)
        metrics_df = trainer.train_models(X_train, X_test, y_train, y_test, self.class_mapping)
        
        # Save metrics
        metrics_df.to_csv(output_dir / 'model_comparison_metrics.csv', index=False)
        print(f"✓ Model comparison saved")
        
        # Plot model comparison
        self._plot_model_comparison(metrics_df, output_dir)
        
        # Use best model
        self.best_model = trainer.best_model
        best_model_name = trainer.best_model_name
        
        y_pred = trainer.y_test_pred_dict[best_model_name]
        y_proba = trainer.y_test_proba_dict[best_model_name]
        
        # Compute metrics
        acc_test = accuracy_score(y_test, y_pred)
        bal_acc_test = balanced_accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        mcc_test = matthews_corrcoef(y_test, y_pred)
        
        try:
            auc_test = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        except:
            auc_test = 0.0
        
        cm = confusion_matrix(y_test, y_pred)
        
        print(f"\n  ===== BEST MODEL: {best_model_name.upper()} =====")
        print(f"  Accuracy: {acc_test:.4f}")
        print(f"  Balanced Accuracy: {bal_acc_test:.4f}")
        print(f"  F1: {f1_test:.4f}")
        print(f"  MCC: {mcc_test:.4f}")
        print(f"  AUROC: {auc_test:.4f}")
        
        # Plot confusion matrix
        print(f"\n  Per-class metrics:")
        for class_idx in range(len(self.class_mapping)):
            class_name = self.class_mapping[class_idx]
            mask = y_test == class_idx
            if mask.sum() > 0:
                class_acc = (y_pred[mask] == class_idx).sum() / mask.sum()
                y_test_binary = (y_test == class_idx).astype(int)
                y_pred_binary = (y_pred == class_idx).astype(int)
                
                prec = precision_score(y_test_binary, y_pred_binary, zero_division=0)
                rec = recall_score(y_test_binary, y_pred_binary, zero_division=0)
                f1 = f1_score(y_test_binary, y_pred_binary, zero_division=0)
                print(f"    {class_name.upper()}: Acc={class_acc:.3f}, Prec={prec:.3f}, Rec={rec:.3f}, F1={f1:.3f} (n={mask.sum()})")
        
        # Plot confusion matrix
        print(f"\n  Plotting confusion matrices...")
        self._plot_confusion_matrix(cm, y_test, y_pred, output_dir)
        
        # Plot ROC curves
        print(f"  Plotting ROC curves...")
        class_names = [self.class_mapping[i].upper() for i in range(len(self.class_mapping))]
        fpr, tpr, auc_score, auc_per_class = self.roc_viz.plot_macro_roc_curve(
            y_test, y_proba, class_names,
            title='Macro-Averaged ROC Curve - Holdout Test Set',
            output_path=output_dir / '02_roc_macro_test_set.png'
        )
        
        self.roc_viz.plot_per_class_roc_curves(
            y_test, y_proba, class_names,
            title='Per-Class ROC Curves - Holdout Test Set',
            output_path=output_dir / '03_roc_per_class_test_set.png'
        )
        
        self.results = {
            'best_model_name': best_model_name,
            'metrics_df': metrics_df,
            'accuracy': acc_test,
            'balanced_accuracy': bal_acc_test,
            'f1': f1_test,
            'mcc': mcc_test,
            'auroc': auc_test,
            'confusion_matrix': cm,
            'y_test': y_test,
            'y_pred': y_pred,
            'y_proba': y_proba,
            'fpr': fpr,
            'tpr': tpr,
            'auc': auc_score,
            'auc_per_class': auc_per_class,
        }
        
        return self.results

    def _plot_model_comparison(self, metrics_df: pd.DataFrame, output_dir: Path):
        """Plot model comparison."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=300)
        
        # Accuracy metrics
        ax = axes[0, 0]
        x = np.arange(len(metrics_df))
        width = 0.15
        
        ax.bar(x - width*1.5, metrics_df['Accuracy'], width, label='Accuracy', alpha=0.8, edgecolor='black')
        ax.bar(x - width*0.5, metrics_df['Balanced_Accuracy'], width, label='Balanced Accuracy', alpha=0.8, edgecolor='black')
        ax.bar(x + width*0.5, metrics_df['Precision'], width, label='Precision', alpha=0.8, edgecolor='black')
        ax.bar(x + width*1.5, metrics_df['Recall'], width, label='Recall', alpha=0.8, edgecolor='black')
        
        ax.set_ylabel('Score', fontweight='bold', fontsize=12)
        ax.set_title('Accuracy Metrics Comparison', fontweight='bold', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
        ax.legend(fontsize=10)
        ax.set_ylim([0, 1.1])
        ax.grid(axis='y', alpha=0.3)
        
        # F1 and MCC
        ax = axes[0, 1]
        ax.bar(x - width/2, metrics_df['F1_Score'], width, label='F1 Score', alpha=0.8, edgecolor='black', color='#2ecc71')
        ax.bar(x + width/2, metrics_df['MCC'], width, label='MCC', alpha=0.8, edgecolor='black', color='#f39c12')
        
        ax.set_ylabel('Score', fontweight='bold', fontsize=12)
        ax.set_title('F1 Score and MCC Comparison', fontweight='bold', fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_df['Model'], rotation=45, ha='right')
        ax.legend(fontsize=10)
        ax.set_ylim([0, 1.1])
        ax.grid(axis='y', alpha=0.3)
        
        # AUROC
        ax = axes[1, 0]
        colors_auc = ['#3498db' if val > 0.7 else '#e74c3c' for val in metrics_df['AUROC']]
        bars = ax.bar(metrics_df['Model'], metrics_df['AUROC'], alpha=0.8, edgecolor='black', linewidth=2, color=colors_auc)
        
        for bar, val in zip(bars, metrics_df['AUROC']):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.02, f'{val:.3f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        ax.set_ylabel('AUROC', fontweight='bold', fontsize=12)
        ax.set_title('AUROC Comparison', fontweight='bold', fontsize=14)
        ax.set_ylim([0, 1.1])
        ax.grid(axis='y', alpha=0.3)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        # Summary table
        ax = axes[1, 1]
        ax.axis('off')
        
        best_idx = metrics_df['Accuracy'].idxmax()
        best_model = metrics_df.iloc[best_idx]
        
        summary_text = f"""
        MULTI-MODEL COMPARISON SUMMARY
        {'='*50}
        
        🏆 BEST MODEL: {best_model['Model'].upper()}
        
        Accuracy:           {best_model['Accuracy']:.4f}
        Balanced Accuracy:  {best_model['Balanced_Accuracy']:.4f}
        Precision:          {best_model['Precision']:.4f}
        Recall:             {best_model['Recall']:.4f}
        F1 Score:           {best_model['F1_Score']:.4f}
        MCC:                {best_model['MCC']:.4f}
        AUROC:              {best_model['AUROC']:.4f}
        
        Models Trained: {len(metrics_df)}
        """
        
        ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, pad=1))
        
        plt.suptitle('Multi-Model Performance Comparison', fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout()
        plt.savefig(output_dir / '01_model_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Model comparison plot saved")

    def _plot_confusion_matrix(self, cm: np.ndarray, y_test, y_pred, output_dir: Path):
        """Plot confusion matrix."""
        class_names = [self.class_mapping[i].upper() for i in range(len(self.class_mapping))]
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
        
        # Counts
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                    xticklabels=class_names, yticklabels=class_names,
                    annot_kws={"size": 12}, linewidths=2)
        axes[0].set_xlabel('Predicted', fontweight='bold', fontsize=12)
        axes[0].set_ylabel('True', fontweight='bold', fontsize=12)
        axes[0].set_title('Confusion Matrix - Holdout Test (Counts)', fontweight='bold', fontsize=13)
        
        # Percentages
        cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', cbar=True, ax=axes[1],
                    xticklabels=class_names, yticklabels=class_names,
                    annot_kws={"size": 12}, cbar_kws={"label": "Percentage (%)"}, linewidths=2)
        axes[1].set_xlabel('Predicted', fontweight='bold', fontsize=12)
        axes[1].set_ylabel('True', fontweight='bold', fontsize=12)
        axes[1].set_title('Confusion Matrix - Holdout Test (%)', fontweight='bold', fontsize=13)
        
        plt.tight_layout()
        plt.savefig(output_dir / '04_confusion_matrix_test_set.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"✓ Test set confusion matrix saved")

    def save_model(self, output_dir: Path):
        """Save model and metadata."""
        model_path = output_dir / "llps_model_lipidation_aware.pkl"
        with open(model_path, 'wb') as f:
            pickle.dump(self.best_model, f)
        print(f"✓ Saved model to {model_path}")
        
        metadata = {
            'class_mapping': self.class_mapping,
            'lipidation_filter': CONFIG['lipidation_filter']
        }
        metadata_path = output_dir / "model_metadata_lipidation_aware.pkl"
        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"✓ Saved metadata to {metadata_path}")

#============================================================================
#MAIN EXECUTION
#============================================================================

def main():
    """Complete workflow - Training and Model Comparison."""
    print("\n" + "="*80)
    print("LLPS CLASSIFICATION WITH LIPIDATION-AWARE FINGERPRINTS")
    print(f"Multi-Model Comparison and Evaluation")
    print(f"Lipidation Filter: {CONFIG['lipidation_filter'].upper()}")
    print("="*80)

    output_dir = Path(CONFIG['output_dir'])
    output_dir.mkdir(exist_ok=True, parents=True)

    # ========== TRAINING WITH MULTI-MODEL COMPARISON ==========
    print("\n" + "="*80)
    print("TRAINING WITH MULTI-MODEL COMPARISON")
    print("="*80)

    classifier = LLPSFingerprintClassifier(random_state=CONFIG['random_state'])
    sequences, labels, metadata = classifier.load_and_prepare_data(CONFIG['training_file'])

    print("\n[4/4] Computing lipidation-aware fingerprints...")
    lipidations = metadata['lipidation'].values
    X = classifier.compute_fingerprints_lipidation_aware(sequences, lipidations)

    training_results = classifier.train_with_holdout_multimodel(X, labels, output_dir)
    classifier.save_model(output_dir)

    # ========== FINAL SUMMARY ==========
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)

    print(f"\n📊 HOLDOUT TEST SET RESULTS:")
    print(f"  Best Model: {training_results['best_model_name']}")
    print(f"  Accuracy: {training_results['accuracy']:.4f}")
    print(f"  Balanced Accuracy: {training_results['balanced_accuracy']:.4f}")
    print(f"  Precision: {training_results['metrics_df'].iloc[0]['Precision']:.4f}")
    print(f"  Recall: {training_results['metrics_df'].iloc[0]['Recall']:.4f}")
    print(f"  F1 Score: {training_results['f1']:.4f}")
    print(f"  MCC: {training_results['mcc']:.4f}")
    print(f"  Macro AUROC: {training_results['auroc']:.4f}")

    print(f"\n📋 MODEL COMPARISON TABLE (Sorted by Accuracy):")
    print(training_results['metrics_df'].to_string(index=False))

    print(f"\n📁 OUTPUT DIRECTORY: {output_dir}")

    print(f"\n📈 OUTPUT FILES:")
    print(f"  • 01_model_comparison.png - Multi-model performance comparison")
    print(f"  • 02_roc_macro_test_set.png - Macro-averaged ROC curve")
    print(f"  • 03_roc_per_class_test_set.png - Per-class ROC curves")
    print(f"  • 04_confusion_matrix_test_set.png - Confusion matrices")
    print(f"  • model_comparison_metrics.csv - Detailed metrics for all models")
    print(f"  • llps_model_lipidation_aware.pkl - Trained best model")
    print(f"  • model_metadata_lipidation_aware.pkl - Model metadata")

    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()